In [0]:
# Define o catálogo e os schemas das camadas Bronze e Silver.
catalog = "meu_catalog"

bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

print(f"Bronze: {bronze_schema}")
print(f"Silver: {silver_schema}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Funções auxiliares para padronizar textos, números, datas e entidades.

def normalizar_texto(coluna):
    """Remove espaços extras e padroniza o texto."""
    return F.regexp_replace(
        F.trim(F.col(coluna).cast("string")),
        r"\s+",
        " "
    )


def numero_limpo(coluna):
    """
    Limpa valores numéricos e monetários em diferentes formatos.
    Converte apenas valores com formato numérico válido.
    """
    c = F.trim(F.col(coluna).cast("string"))

    c = F.when(
        c.isNull()
        | F.lower(c).isin(
            "unknown", "não informado", "nao informado",
            "null", "none", "n/a", "na", "nan", ""
        ),
        None
    ).otherwise(c)

    # Remove moedas, símbolos e espaços.
    c = F.regexp_replace(c, r"(?i)usd", "")
    c = F.regexp_replace(c, r"(?i)^r\$?", "")
    c = F.regexp_replace(c, r"[$€£]", "")
    c = F.regexp_replace(c, r"\s+", "")

    # Identifica multiplicadores abreviados.
    fator = (
        F.when(c.rlike(r"[kK]$"), F.lit(1000))
         .when(c.rlike(r"[mM]$"), F.lit(1000000))
         .otherwise(F.lit(1))
    )
    c = F.regexp_replace(c, r"[kKmM]$", "")

    # Converte formatos com separadores de milhar e decimal.
    c = F.when(
        c.rlike(r"^-?\d{1,3}(\.\d{3})+,\d+$"),
        F.regexp_replace(
            F.regexp_replace(c, r"\.", ""),
            ",",
            "."
        )
    ).otherwise(c)

    c = F.when(
        c.rlike(r"^-?\d{1,3}(,\d{3})+\.\d+$"),
        F.regexp_replace(c, ",", "")
    ).otherwise(c)

    c = F.when(
        c.rlike(r"^-?\d{1,3}(,\d{3})+$"),
        F.regexp_replace(c, ",", "")
    ).when(
        c.rlike(r"^-?\d+,\d+$"),
        F.regexp_replace(c, ",", ".")
    ).otherwise(c)

    # Converte somente valores numéricos válidos.
    valido = c.rlike(r"^-?\d+(\.\d+)?$")

    return (
        F.when(
            valido,
            c.cast("decimal(38,10)") * fator
        )
        .otherwise(F.lit(None).cast("decimal(38,10)"))
    )


def inteiro_seguro(coluna):
    """Converte valores numéricos inteiros e invalida valores fracionários."""
    n = numero_limpo(coluna).cast("double")
    return (
        F.when(
            n.isNotNull() & (n == F.floor(n)),
            n.cast("int")
        )
        .otherwise(F.lit(None).cast("int"))
    )


def data_segura(coluna, formato, regex):
    """Converte a data somente quando o formato informado for válido."""
    c = F.trim(F.col(coluna).cast("string"))
    return F.when(
        c.rlike(regex),
        F.to_date(c, formato)
    ).otherwise(F.lit(None).cast("date"))


def completude(colunas):
    """Conta quantos campos preenchidos o registro possui."""
    score = F.lit(0)
    for coluna in colunas:
        preenchido = (
            F.col(coluna).cast("string").isNotNull()
            & (F.trim(F.col(coluna).cast("string")) != "")
        )
        score = score + F.when(preenchido, 1).otherwise(0)
    return score


def hash_deterministico(colunas):
    """Gera um hash estável para desempate entre registros."""
    valores = [
        F.coalesce(F.col(coluna).cast("string"), F.lit(""))
        for coluna in colunas
    ]
    return F.sha2(F.concat_ws("\u001f", *valores), 256)


def popularidade_limpa(coluna):
    """Converte a popularidade usando vírgula como separador decimal."""
    c = F.regexp_replace(F.trim(F.col(coluna).cast("string")), ",", ".")
    return F.when(c.rlike(r"^-?\d+(\.\d+)?$"), c.cast("double"))


def adicionar_data_multiformato(df, coluna, destino):
    """Converte datas da origem nos formatos ISO, barra ou hífen."""
    c = F.trim(F.col(coluna).cast("string"))
    df = (
        df.withColumn("_d_iso", F.regexp_extract(c, r"^(\d{4}-\d{1,2}-\d{1,2})", 1))
          .withColumn("_d_barra", F.when(c.rlike(r"^\d{1,2}/\d{1,2}/\d{4}$"), c))
          .withColumn("_d_hifen", F.when(c.rlike(r"^\d{1,2}-\d{1,2}-\d{4}$"), c))
    )
    return (
        df.withColumn(destino, F.coalesce(
            F.expr("try_to_date(nullif(_d_iso, ''), 'yyyy-M-d')"),
            F.expr("try_to_date(_d_barra, 'd/M/yyyy')"),
            F.expr("try_to_date(_d_hifen, 'M-d-yyyy')"),
        ))
        .drop("_d_iso", "_d_barra", "_d_hifen")
    )


def entidade_valida(coluna):
    """Filtra valores que não correspondem a nomes de entidades válidos."""
    c = F.col(coluna)
    return (
        (c != "")
        & ~F.lower(c).isin("unknown", "null", "none", "nan", "na", "n/a", "[]", "nenhum")
        & ~c.rlike(r"^\d+(\.\d+)?$")
        & ~c.rlike(r"^/.*\.(jpg|jpeg|png)$")
        & ~c.rlike(r"\\")
        & (F.length(c) <= 60)
        & ~(c.rlike(r"[.!?]$") & (F.size(F.split(c, " ")) >= 5))
    )

In [0]:
# Regras: normalizar e traduzir status, manter a ingestão mais recente,
# converter datas e derivar o ano de lançamento.

# Permite executar a célula isoladamente.
if "bronze_schema" not in globals():
    catalog = "meu_catalog"
    bronze_schema = f"{catalog}.bronze"
    silver_schema = f"{catalog}.silver"

df_info_raw = spark.table(f"{bronze_schema}.tb_movies_info")

info_cols = [
    "title", "original_title", "release_date", "runtime",
    "original_language", "status", "overview", "tagline"
]

# Define os critérios para escolher o registro mais recente e completo.
w_info = Window.partitionBy("id").orderBy(
    F.col("ingestion_datetime").desc_nulls_last(),
    F.col("_completude").desc(),
    F.col("_titulo_capitalizado").desc(),
    F.col("_hash").asc()
)

# Padroniza o status antes da tradução.
status = F.regexp_replace(F.lower(F.trim(F.col("status"))), r"[-_]+", " ")
status = F.regexp_replace(status, r"\s+", " ")

df_info = (
    df_info_raw
    .withColumn("_completude", completude(info_cols))
    .withColumn("_hash", hash_deterministico(info_cols))
    .withColumn(
        "_titulo_capitalizado",
        F.coalesce(F.col("title") != F.lower(F.col("title")), F.lit(False))
    )
    .withColumn("_status", status)
    .withColumn(
        "_status_pt",
        F.when(F.col("_status") == "released", "Lançado")
         .when(F.col("_status") == "post production", "Pós-Produção")
         .when(F.col("_status") == "in production", "Em Produção")
         .when(F.col("_status") == "planned", "Planejado")
         .when(F.col("_status") == "rumored", "Rumores")
         .when(F.col("_status") == "canceled", "Cancelado")
         .otherwise("Não Informado")
    )
    .withColumn("_rn", F.row_number().over(w_info))
    .filter(F.col("_rn") == 1)
)

# Converte as datas conforme os formatos encontrados na origem.
df_info = adicionar_data_multiformato(df_info, "release_date", "data_lancamento")

df_info = (
    df_info
    # Deriva o ano a partir da data de lançamento.
    .withColumn("ano_lancamento", F.year("data_lancamento"))
    .select(
        F.col("id").cast("string").alias("id_filme"),
        normalizar_texto("title").cast("string").alias("titulo"),
        normalizar_texto("original_title").cast("string").alias("titulo_original"),
        F.col("data_lancamento").cast("date"),
        F.col("ano_lancamento").cast("int"),
        inteiro_seguro("runtime").alias("duracao_minutos"),
        normalizar_texto("original_language").cast("string").alias("idioma_original"),
        F.col("_status_pt").cast("string").alias("status_filme"),
        normalizar_texto("overview").cast("string").alias("sinopse"),
        normalizar_texto("tagline").cast("string").alias("frase_divulgacao")
    )
)

# Grava a tabela Silver em formato Delta.
df_info.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{silver_schema}.tb_info_filmes")

print(f"{silver_schema}.tb_info_filmes criada")
display(df_info.limit(10))

In [0]:
df = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

# Converte os campos da API para tipos adequados e cria a data da cotação.
df_cot = (
    df.select(
        F.to_timestamp("dataHoraCotacao").alias("data_hora_cotacao"),
        numero_limpo("cotacaoCompra").cast("decimal(18,6)").alias("cotacao_compra")
    )
    .filter(F.col("data_hora_cotacao").isNotNull())
    .filter(F.col("cotacao_compra").isNotNull())
    .withColumn("data_cotacao", F.to_date("data_hora_cotacao"))
)

# Quando houver mais de uma cotação no mesmo dia, mantém a mais recente.
w_dia = Window.partitionBy("data_cotacao").orderBy(
    F.col("data_hora_cotacao").desc()
)

df_diaria = (
    df_cot
    .withColumn("_rn", F.row_number().over(w_dia))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# Identifica o primeiro e o último dia disponíveis na série.
limites = df_diaria.select(
    F.min("data_cotacao").alias("data_min"),
    F.max("data_cotacao").alias("data_max")
).first()

if limites["data_min"] is None:
    raise ValueError("bronze.tb_cotacao_dolar não possui cotações válidas.")

# Cria uma sequência diária entre a primeira e a última cotação.
# Isso inclui finais de semana e feriados, que não possuem cotação na API.
df_datas = spark.sql(f"""
    SELECT explode(sequence(
        to_date('{limites["data_min"]}'),
        to_date('{limites["data_max"]}'),
        interval 1 day
    )) AS data_cotacao
""")

# Junta as cotações reais à sequência completa de datas.
df_cot_silver = (
    df_datas
    .join(
        df_diaria.select(
            "data_cotacao",
            "data_hora_cotacao",
            "cotacao_compra"
        ),
        "data_cotacao",
        "left"
    )
)

# Define uma janela acumulada para buscar a última cotação disponível.
w_ffill = Window.orderBy("data_cotacao").rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)

df_cot_silver = (
    df_cot_silver
    # Preenche dias sem cotação com o último valor disponível.
    .withColumn(
        "cotacao_compra",
        F.last("cotacao_compra", ignorenulls=True).over(w_ffill)
    )
    .select(
        "data_cotacao",
        "data_hora_cotacao",
        F.col("cotacao_compra").cast("decimal(18,6)")
    )
)

# Salva a série contínua de cotações na camada Silver.
df_cot_silver.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{silver_schema}.tb_cotacao_dolar")

display(df_cot_silver)

In [0]:
df = spark.table(f"{bronze_schema}.tb_movies_financials")

fin_cols = ["budget", "revenue"]

# Limpa os valores antes da deduplicação para medir corretamente a completude.
df_fin_limpo = (
    df
    .withColumn("_hash", hash_deterministico(fin_cols))
    .select(
        F.col("id").cast("string").alias("id_filme"),
        "ingestion_datetime",
        "_hash",
        numero_limpo("budget").cast("decimal(18,2)").alias("_orcamento"),
        numero_limpo("revenue").cast("decimal(18,2)").alias("_receita")
    )
    # Valores zerados ou negativos são considerados ausentes.
    .withColumn("orcamento_usd", F.when(F.col("_orcamento") > 0, F.col("_orcamento")))
    .withColumn("receita_usd", F.when(F.col("_receita") > 0, F.col("_receita")))
    .drop("_orcamento", "_receita")
    .withColumn(
        "_completude",
        F.col("orcamento_usd").isNotNull().cast("int")
        + F.col("receita_usd").isNotNull().cast("int")
    )
)

# Mantém um registro por filme, priorizando a ingestão mais recente e os dados mais completos.
w_fin = Window.partitionBy("id_filme").orderBy(
    F.col("ingestion_datetime").desc_nulls_last(),
    F.col("_completude").desc(),
    F.col("_hash").asc()
)

df_fin = (
    df_fin_limpo
    .withColumn("_rn", F.row_number().over(w_fin))
    .filter(F.col("_rn") == 1)
    .select("id_filme", "orcamento_usd", "receita_usd")
)

# Obtém a cotação mais recente disponível na camada Silver.
taxa = (
    spark.table(f"{silver_schema}.tb_cotacao_dolar")
    .filter(F.col("cotacao_compra").isNotNull())
    .orderBy(F.col("data_cotacao").desc())
    .select(F.col("cotacao_compra").alias("taxa_dolar_brl"))
    .first()
)

if taxa is None:
    raise ValueError("Não foi possível obter uma cotação válida da Silver.")

taxa_dolar = taxa["taxa_dolar_brl"]

df_fin = (
    df_fin
    # Converte orçamento e receita de dólar para real usando a cotação obtida.
    .withColumn(
        "orcamento_brl",
        F.round(F.col("orcamento_usd") * F.lit(taxa_dolar), 2)
        .cast("decimal(18,2)")
    )
    .withColumn(
        "receita_brl",
        F.round(F.col("receita_usd") * F.lit(taxa_dolar), 2)
        .cast("decimal(18,2)")
    )
    # Calcula o lucro somente quando receita e orçamento estão disponíveis.
    .withColumn(
        "lucro_usd",
        F.when(
            F.col("receita_usd").isNotNull()
            & F.col("orcamento_usd").isNotNull(),
            F.round(
                F.col("receita_usd") - F.col("orcamento_usd"),
                2
            )
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "lucro_brl",
        F.when(
            F.col("receita_brl").isNotNull()
            & F.col("orcamento_brl").isNotNull(),
            F.round(
                F.col("receita_brl") - F.col("orcamento_brl"),
                2
            )
        ).cast("decimal(18,2)")
    )
    # Calcula a margem evitando divisão por zero ou valores ausentes.
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("receita_usd").isNotNull()
            & (F.col("receita_usd") != 0)
            & F.col("lucro_usd").isNotNull(),
            F.round(
                (F.col("lucro_usd") / F.col("receita_usd")) * 100,
                2
            )
        ).cast("decimal(10,2)")
    )
    .select(
        "id_filme", "orcamento_usd", "receita_usd",
        "orcamento_brl", "receita_brl",
        "lucro_usd", "lucro_brl",
        "margem_lucro_percentual"
    )
)

# Grava os dados financeiros tratados na camada Silver.
df_fin.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{silver_schema}.tb_financeiro_filmes")

print(f"Cotação aplicada: {taxa_dolar}")
display(df_fin.limit(10))

In [0]:
df = spark.table(f"{bronze_schema}.tb_movies_metrics")

# Define as colunas de métricas usadas no tratamento e na deduplicação.
metric_cols = [
    "popularity", "vote_average", "vote_count",
    "averageRating", "numVotes"
]

# Limpa cada métrica usando o tipo de conversão adequado.
# Para popularidade, a vírgula é tratada como separador decimal.
pop = popularidade_limpa("popularity")
tmdb = numero_limpo("vote_average").cast("double")
v_tmdb = inteiro_seguro("vote_count")
imdb = numero_limpo("averageRating").cast("double")
v_imdb = inteiro_seguro("numVotes")

# Limpa antes da deduplicação para que a completude considere apenas valores válidos.
df_metrics_limpo = (
    df
    .withColumn("_hash", hash_deterministico(metric_cols))
    .select(
        F.col("id").cast("string").alias("id_filme"),
        "ingestion_datetime",
        "_hash",
        # Popularidade negativa é considerada inválida.
        F.when(pop >= 0, pop).alias("popularidade"),
        # Notas fora da escala de 0 a 10 são consideradas inválidas.
        F.when((tmdb >= 0) & (tmdb <= 10), tmdb).alias("nota_media_tmdb"),
        F.when(v_tmdb >= 0, v_tmdb).alias("qtd_votos_tmdb"),
        F.when((imdb >= 0) & (imdb <= 10), imdb).alias("nota_media_imdb"),
        F.when(v_imdb >= 0, v_imdb).alias("qtd_votos_imdb")
    )
)

colunas_finais = [
    "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
    "nota_media_imdb", "qtd_votos_imdb"
]

# Calcula a quantidade de métricas válidas para usar como critério de desempate.
completude_limpa = F.lit(0)
for c in colunas_finais:
    completude_limpa = completude_limpa + F.col(c).isNotNull().cast("int")

# Prioriza a ingestão mais recente, depois os dados mais completos e, por fim, o hash.
w_metrics = Window.partitionBy("id_filme").orderBy(
    F.col("ingestion_datetime").desc_nulls_last(),
    F.col("_comp").desc(),
    F.col("_hash").asc()
)

df_metrics = (
    df_metrics_limpo
    .withColumn("_comp", completude_limpa)
    .withColumn("_rn", F.row_number().over(w_metrics))
    .filter(F.col("_rn") == 1)
    .select("id_filme", *colunas_finais)
)

# Grava as métricas tratadas na tabela Silver.
df_metrics.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

display(df_metrics.limit(10))

In [0]:
df = spark.table(f"{bronze_schema}.tb_movies_reviews")

# Converte a nota para número antes de validar a escala.
nota = numero_limpo("nota").cast("double")

df_reviews = (
    df.select(
        F.col("id").cast("string").alias("id_filme"),
        normalizar_texto("nome").cast("string").alias("nome_usuario"),
        # Aceita notas de 0 a 10; valores fora da escala são convertidos para NULL.
        F.when(
            (nota >= 0) & (nota <= 10),
            nota
        ).otherwise(
            F.lit(None).cast("double")
        ).alias("nota_usuario"),
        # Substitui comentários ausentes ou vazios por um texto padrão.
        F.when(
            F.col("comentario").isNull()
            | (F.trim(F.col("comentario")) == ""),
            F.lit("Sem comentário")
        ).otherwise(
            normalizar_texto("comentario").cast("string")
        ).alias("comentario_usuario")
    )
    # Remove avaliações completamente duplicadas.
    .dropDuplicates([
        "id_filme",
        "nome_usuario",
        "nota_usuario",
        "comentario_usuario"
    ])
)

# Grava as avaliações tratadas na tabela Silver.
df_reviews.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

display(df_reviews.limit(10))

In [0]:
df = spark.table(f"{bronze_schema}.tb_credits_and_tags")

# Padroniza os separadores para permitir a divisão dos gêneros em registros individuais.
genres = F.regexp_replace(
    F.coalesce(F.col("genres"), F.lit("")),
    r"[;,|]+",
    ","
)
genres = F.regexp_replace(genres, r",+", ",")

df_generos = (
    df.select(
        F.col("id").cast("string").alias("id_filme"),
        F.explode(F.split(genres, ",")).alias("nome_genero")
    )
    .withColumn("nome_genero", normalizar_texto("nome_genero"))
    .filter(F.col("nome_genero") != "")
    # Remove valores numéricos que não representam gêneros.
    .filter(~F.col("nome_genero").rlike(r"^\d+(\.\d+)?$"))
    # Remove valores nulos, placeholders e nomes de outros campos presentes na origem.
    .filter(~F.lower(F.col("nome_genero")).isin(
        "unknown", "null", "none", "nan", "na", "n/a",
        "production companies", "production countries",
        "spoken languages", "keywords", "cast", "directors", "writers"
    ))
    # Mantém apenas uma ocorrência de cada gênero por filme.
    .dropDuplicates(["id_filme", "nome_genero"])
)

# Grava os gêneros tratados na tabela Silver.
df_generos.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{silver_schema}.tb_generos")

display(df_generos.limit(20))

In [0]:
df = spark.table(f"{bronze_schema}.tb_credits_and_tags")

# Define a taxonomia válida de gêneros para eliminar valores incorretos da origem.
GENEROS_VALIDOS = {
    g.lower(): g for g in [
        "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary",
        "Drama", "Family", "Fantasy", "History", "Horror", "Music", "Mystery",
        "Romance", "Science Fiction", "TV Movie", "Thriller", "War", "Western"
    ]
}
mapa_generos = F.create_map(
    *[x for k, v in GENEROS_VALIDOS.items() for x in (F.lit(k), F.lit(v))]
)

# Padroniza os separadores antes de dividir a coluna em gêneros individuais.
genres = F.regexp_replace(
    F.coalesce(F.col("genres"), F.lit("")),
    r"[;,|]+",
    ","
)
genres = F.regexp_replace(genres, r",+", ",")

df_generos = (
    df.select(
        F.col("id").cast("string").alias("id_filme"),
        F.explode(F.split(genres, ",")).alias("nome_genero")
    )
    .withColumn("nome_genero", normalizar_texto("nome_genero"))
    # Normaliza maiúsculas/minúsculas e mantém apenas gêneros da taxonomia válida.
    .withColumn("nome_genero", mapa_generos[F.lower(F.col("nome_genero"))])
    .filter(F.col("nome_genero").isNotNull())
    # Garante uma ocorrência de cada gênero por filme.
    .dropDuplicates(["id_filme", "nome_genero"])
)

# Grava os gêneros tratados na tabela Silver.
df_generos.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{silver_schema}.tb_generos")

display(df_generos.limit(20))

In [0]:
df = spark.table(f"{bronze_schema}.tb_credits_and_tags")

def entidades(coluna, tipo):
    # Padroniza os separadores antes de dividir os valores em registros individuais.
    valores = F.regexp_replace(
        F.coalesce(F.col(coluna), F.lit("")),
        r"[;,|]+",
        ","
    )
    valores = F.regexp_replace(valores, r",+", ",")

    return (
        df.select(
            F.col("id").cast("string").alias("id_filme"),
            F.explode(F.split(valores, ",")).alias("nome_entidade")
        )
        .withColumn("nome_entidade", normalizar_texto("nome_entidade"))
        # Remove valores que não representam entidades válidas.
        .filter(entidade_valida("nome_entidade"))
        # Padroniza a capitalização dos nomes.
        .withColumn("nome_entidade", F.initcap(F.col("nome_entidade")))
        .withColumn("tipo_entidade", F.lit(tipo))
    )

# Trata pessoas e empresas sem tentar reconstruir automaticamente o Column Shift.
df_pessoas = (
    entidades("cast", "Ator")
    .unionByName(entidades("directors", "Diretor"))
    .unionByName(entidades("writers", "Roteirista"))
    .unionByName(entidades("production_companies", "Produtora"))
    # Remove registros duplicados dentro do mesmo filme e tipo de entidade.
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

# Grava pessoas e empresas tratadas na tabela Silver.
df_pessoas.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{silver_schema}.tb_pessoas_empresas")

display(df_pessoas.limit(20))


In [0]:
tabelas = [
    "tb_info_filmes",
    "tb_financeiro_filmes",
    "tb_metricas_engajamento",
    "tb_avaliacoes_usuarios",
    "tb_generos",
    "tb_pessoas_empresas",
    "tb_cotacao_dolar"
]

# Lista as tabelas disponíveis na camada Silver.
display(spark.sql(f"SHOW TABLES IN {silver_schema}"))

for tabela in tabelas:
    dataframe = spark.table(f"{silver_schema}.{tabela}")
    print(f"\n=== {tabela} ===")
    print(f"Linhas: {dataframe.count()}")
    print("Colunas:", dataframe.columns)
    dataframe.printSchema()

# Verifica se as tabelas que devem ter uma linha por filme possuem duplicidades.
print("\nDuplicidades por id — financeiro:")
display(
    spark.table(f"{silver_schema}.tb_financeiro_filmes")
    .groupBy("id_filme")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicidades por id — métricas:")
display(
    spark.table(f"{silver_schema}.tb_metricas_engajamento")
    .groupBy("id_filme")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicidades por id — info:")
display(
    spark.table(f"{silver_schema}.tb_info_filmes")
    .groupBy("id_filme")
    .count()
    .filter(F.col("count") > 1)
)

In [0]:
from pyspark.sql import functions as F

# Obtém os IDs distintos das tabelas que devem se relacionar pelo filme.
info = spark.table(f"{silver_schema}.tb_info_filmes").select("id_filme").distinct()
financeiro = spark.table(f"{silver_schema}.tb_financeiro_filmes").select("id_filme").distinct()
metricas = spark.table(f"{silver_schema}.tb_metricas_engajamento").select("id_filme").distinct()

# Verifica se existem registros sem correspondência entre as tabelas relacionadas.
print("=== Integridade referencial ===")
print(f"Financeiro sem info: {financeiro.join(info, on='id_filme', how='left_anti').count()}")
print(f"Métricas sem info: {metricas.join(info, on='id_filme', how='left_anti').count()}")
print(f"Info sem financeiro: {info.join(financeiro, on='id_filme', how='left_anti').count()}")
print(f"Info sem métricas: {info.join(metricas, on='id_filme', how='left_anti').count()}")

# Verifica se existem IDs nulos nas tabelas principais.
for tabela in ["tb_info_filmes", "tb_financeiro_filmes", "tb_metricas_engajamento"]:
    qtd_null = spark.table(f"{silver_schema}.{tabela}").filter(F.col("id_filme").isNull()).count()
    print(f"{tabela}: {qtd_null} id_filme nulos")

# Verifica se existe mais de uma cotação para o mesmo dia.
print("Duplicidades por data - cotação:")
display(
    spark.table(f"{silver_schema}.tb_cotacao_dolar")
    .groupBy("data_cotacao")
    .count()
    .filter(F.col("count") > 1)
)

# Valida se os gêneros estão dentro do domínio esperado.
print("Gêneros distintos (esperado: <= 19):")
print(spark.table(f"{silver_schema}.tb_generos").select("nome_genero").distinct().count())

# Verifica se a popularidade está dentro de uma ordem de grandeza esperada.
print("Popularidade máxima (esperado: ~3 mil, não centenas de milhar):")
display(spark.table(f"{silver_schema}.tb_metricas_engajamento").agg(F.max("popularidade")))

# Verifica se datas numéricas da origem foram convertidas corretamente.
print("Datas de lançamento nulas com data de origem preenchida (esperado: 0):")
display(
    spark.table(f"{bronze_schema}.tb_movies_info").alias("b")
    .join(spark.table(f"{silver_schema}.tb_info_filmes").alias("s"),
          F.col("b.id") == F.col("s.id_filme"))
    .filter(F.col("s.data_lancamento").isNull() & F.col("b.release_date").rlike(r"^\s*\d"))
    .count()
)